In [1]:
# This is the python script performing the analysis of many LC-MS files for RiPP modifications
import pandas as pd
import pickle
from pathlib import Path
from plotting_analysis_functions import *
import re

In [2]:
# --- 1. Pre-flight Compatibility Formatting ---
# NOTE: master-dataframe construction now lives in plotting_analysis_functions.py as
# build_master_dataframe_from_groups(), sourced directly from each group's curated
# ReplicateGroupResults (*_manually_reviewed.pkl.gz) via generate_modification_state_matrix(),
# rather than from the older '_CURATED_APPROVED.pkl' report format. See the next cell.

def prepare_dataframe_for_plotting(df: pd.DataFrame) -> pd.DataFrame:
    """
    Guarantees that columns and categorical fields perfectly match the exact
    expectations of your custom calculation and alignment plotting functions.
    """
    plot_df = df.copy()

    # 1. Map alternative column names to match your strict function expectations
    column_mappings = {
        'group_name': 'experiment_id',       # Maps group_name to experiment_id
        'peak_area': 'area',
        'target_sequence': 'peptide_sequence',
        'cluster_id': 'cut_id'
    }

    for old_col, new_col in column_mappings.items():
        if old_col in plot_df.columns and new_col not in plot_df.columns:
            plot_df[new_col] = plot_df[old_col]

    # Ensure 'target_sequence' exists (keep a copy of the unmodified peptide string)
    if 'target_sequence' not in plot_df.columns and 'peptide_sequence' in plot_df.columns:
        plot_df['target_sequence'] = plot_df['peptide_sequence']

    # 2. Since this data is already curated (only manually-approved peaks make it into
    # generate_modification_state_matrix()), treat everything as accepted 'Gold' if the
    # 'tier' column isn't explicitly present.
    if 'tier' not in plot_df.columns:
        plot_df['tier'] = 'Gold'

    # 3. Handle modification string conversions expected by your calculator loop
    if 'target_modifications' in plot_df.columns:
        # Standardize string format to look like your check: "{'unmodified': None}"
        plot_df['modification_state'] = plot_df['target_modifications'].astype(str).replace({
            "{'unmodified': None}": "{'unmodified': None}",
            "unmodified": "{'unmodified': None}"
        })
    elif 'modification_state' not in plot_df.columns:
        plot_df['modification_state'] = "{'unmodified': None}"

    # 4. Final safety fallbacks for columns that absolutely must exist
    required_cols = ['experiment_id', 'sample_id', 'peptide_sequence', 'cut_id', 'enzyme', 'precursor_name', 'area']
    for col in required_cols:
        if col not in plot_df.columns:
            plot_df[col] = 'N/A' if col != 'area' else 0.0

    return plot_df

In [1]:
from pathlib import Path
from plotting_analysis_functions import build_master_dataframe_from_groups

# --- Configuration ---
REPORTS_PATH = r'D:\Spliceotide_Data_Analysis\output_reports_plates'

# Build the master DataFrame directly from every group's curated ReplicateGroupResults
# (one *_manually_reviewed.pkl.gz per group, produced by peak_reviewing_pipeline.py). Experiment
# metadata (required_grouping_cols, sample_id, precursor_name, peptide_sequence, core
# coordinates, etc.) is pulled straight from each replicate's MSExperiment object, so no separate
# metadata Excel merge is needed anymore.
master_df = build_master_dataframe_from_groups(REPORTS_PATH)

Found 140 manually reviewed group file(s) under: D:\Spliceotide_Data_Analysis\output_reports_plates
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_10_HypX-HypY_TJ-C-689_24_TEV\point_mutants_precursor_10_HypX-HypY_TJ-C-689_24_TEV_manually_reviewed.pkl.gz
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_10_PlpX-PlpY_TJ-C-559_24_TEV\point_mutants_precursor_10_PlpX-PlpY_TJ-C-559_24_TEV_manually_reviewed.pkl.gz
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_11_HypX-HypY_TJ-C-689_24_TEV\point_mutants_precursor_11_HypX-HypY_TJ-C-689_24_TEV_manually_reviewed.pkl.gz
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_11_PlpX-PlpY_TJ-C-559_24_TEV\point_mutants_precursor_11_PlpX-PlpY_TJ-C-559_24_TEV_manua

In [2]:
# --- SI report generation: single example first ---
# Sanity-check the SI report machinery on ONE group before running the full batch in the next
# cell. This mirrors exactly what generate_si_reports() does per group internally:
#   1. Load a curated ReplicateGroupResults.
#   2. Build a GroupedExperiments via from_replicate_group_results().
#   3. select_best_cut_charge() -- fewest stub replicates first, then area, then quality.
#   4. generate_si_report() -- single-page A4 PDF, written beside the source pickle.
# Worth opening the resulting PDF and eyeballing it before committing to the whole batch below.
from analysis_containers import ReplicateGroupResults, GroupedExperiments
from plotting_analysis_functions import find_group_result_files

example_group_files = find_group_result_files(REPORTS_PATH)
print(f"Found {len(example_group_files)} manually reviewed group file(s); using the first as an example.")

example_path = example_group_files[2]
example_results = ReplicateGroupResults.load(example_path)
example_grouped = GroupedExperiments.from_replicate_group_results(example_results)

example_cut, example_charge = example_grouped.select_best_cut_charge()
print(f"Best (cut, charge) for {example_results.group_name}: {example_cut}, +{example_charge}")

example_pdf_path = example_grouped.generate_si_report(
    example_path.parent,
    plot_mz=True,
    plot_ms2=False,
    dpi=400,
    page_number=1,
    source_path=example_path,
)
print(f"Wrote example SI report to: {example_pdf_path}")

Found 140 manually reviewed group file(s); using the first as an example.
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_11_HypX-HypY_TJ-C-689_24_TEV\point_mutants_precursor_11_HypX-HypY_TJ-C-689_24_TEV_manually_reviewed.pkl.gz
Best (cut, charge) for point_mutants_precursor_11_HypX-HypY_TJ-C-689_24_TEV: (0, 16), +1
Spectral data for 'plate_1_C2_point_mutants' successfully loaded from cache.
Wrote example SI report to: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_11_HypX-HypY_TJ-C-689_24_TEV\001_point_mutants_precursor_11_HypX-HypY_TJ-C-689_24_TEV_SI_Report_Cut_(0,_16)_z1.pdf


In [3]:
# --- Generate all SI data PDF reports, then combine them into one document ---
# For every group's curated ReplicateGroupResults (*_manually_reviewed.pkl.gz):
#   1. Builds a GroupedExperiments via GroupedExperiments.from_replicate_group_results().
#   2. Picks the single best (fragment_cut, charge) via select_best_cut_charge() -- fewest stub
#      replicates first, then highest average area, then highest average quality_score (mirrors
#      extract_best_cut_charge()'s priority above).
#   3. Plots only the manually curated (selected_peaks) peaks at that cut/charge; replicates with
#      no curated selection for a given modification state are listed in the summary table as
#      stubs but are not plotted.
#   4. Writes one single-page, high-resolution A4 PDF per group -- with headers/footers, a
#      sequential page number, and a page-order-matching filename -- into the SAME folder as that
#      group's pickle (not a separate output tree).
# Finally, merges every generated PDF, in that same page order, into one combined document at
# COMBINED_SI_PDF_PATH.
from plotting_analysis_functions import generate_and_combine_si_reports

COMBINED_SI_PDF_PATH = r'D:\Spliceotide_Data_Analysis\output_reports_plates\SI_reports\combined_SI_report.pdf'

combined_si_pdf_path = generate_and_combine_si_reports(
    REPORTS_PATH, COMBINED_SI_PDF_PATH, plot_mz=True, plot_ms2=False, dpi=400
)

Found 140 manually reviewed group file(s) under: D:\Spliceotide_Data_Analysis\output_reports_plates
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_10_HypX-HypY_TJ-C-689_24_TEV\point_mutants_precursor_10_HypX-HypY_TJ-C-689_24_TEV_manually_reviewed.pkl.gz
Spectral data for 'plate_1_B2_point_mutants' successfully loaded from cache.
  Wrote: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_10_HypX-HypY_TJ-C-689_24_TEV\001_point_mutants_precursor_10_HypX-HypY_TJ-C-689_24_TEV_SI_Report_Cut_(0,_16)_z2.pdf
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_10_PlpX-PlpY_TJ-C-559_24_TEV\point_mutants_precursor_10_PlpX-PlpY_TJ-C-559_24_TEV_manually_reviewed.pkl.gz
Spectral data for 'plate_1_B7_point_mutants' successfully loaded from cache.
  Wrote: D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_10_

In [ ]:
from plotting_analysis_functions import extract_best_cut_charge, collapse_modification_replicates

# --- 2. Data Preparation Chain ---
# Step A: For every experimental group, pick the single best (cut, charge) combination
# (highest average quality_score across replicates/modifications, then highest average area).
best_cut_charge_df = extract_best_cut_charge(master_df)

# Step B: Collapse each replicate down to its unmodified peptide, computing
# total_modification_ratio (aliased as mod_ratio) = unmodified area / overall_signal_intensity.
processed_ratios_df = collapse_modification_replicates(best_cut_charge_df)

print(f"Summary data ready. Calculated modification ratios for {len(processed_ratios_df)} features.")

In [ ]:
# matplotlib inline

point_mutant_comparison = processed_ratios_df[processed_ratios_df["experiment_id"] == "point_mutants"]


# we plot
point_mutant_plot = point_mutant_comparison[
    (point_mutant_comparison["precursor_plasmid"].str.contains("precursor")) |
    (point_mutant_comparison["precursor_plasmid"] == "TJ-C-706") |
    (point_mutant_comparison["precursor_plasmid"] == "TJ-C-710")
]
# if precursor_plasmid is 706 or 710 then precursor_name is artificial-2
# else it is the same as precursor_plasmid but with _ replaced by -
point_mutant_plot["precursor_name"] = np.where(
    point_mutant_plot["precursor_plasmid"].isin(["TJ-C-706", "TJ-C-710"]),
    "artificial-2",
    point_mutant_plot["precursor_plasmid"].str.replace("precursor_", "mutant-"))

REFERENCE_SEQUENCE = "GGCGPQPVYGVILPWA"
create_grouped_alignment_plot(
    point_mutant_plot,
    ref_sequence = REFERENCE_SEQUENCE,
    metadata_cols= ['precursor_name'],
    color_scheme_name= "hydrophobicity",
    show_ddg= False,
    preset = 'poster',
    use_alignment = True,
    drop_identical = True,
    sequence_column = "target_sequence",
    metatitle="Core",
    seqtitle="Point Mutant Sequence",
    suptitle="Modification Efficiencies of Point Mutants by Enzyme",
    savepath=r"summary_data_files/point_mutant_plates_plot_poster.eps",
    plot_baseline=True
)


In [ ]:

rosetta_comparison = processed_ratios_df[processed_ratios_df["experiment_id"] == "rosetta_top_47"]
rosetta_comparison["precursor_name"] = np.where(
    rosetta_comparison["precursor_plasmid"].isin(["TJ-C-710"]),
    "artificial-2",
    rosetta_comparison["precursor_plasmid"].str.replace("precursor_", "designed-"))


# If your dataset spans multiple distinct enzymes (e.g., different maturities or conditions):
REFERENCE_SEQUENCE = "GGCGPQPVYGVILPWA"

REFERENCE_SEQUENCE = "GGCGPQPVYGVILPWA"
create_grouped_alignment_plot(
    rosetta_comparison,
    ref_sequence = REFERENCE_SEQUENCE,
    metadata_cols= ['precursor_name'],
    color_scheme_name= "hydrophobicity",
    show_ddg= False,
    preset = 'poster',
    use_alignment = True,
    drop_identical = True,
    sequence_column = "peptide_sequence_meta",
    metatitle="Core",
    seqtitle="Designed Cores",
    suptitle="Modification Efficiency of Rosetta Designed Peptides",
    savepath=r"summary_data_files/rosetta_designs_plates_plot.eps",
    plot_baseline=True
)


In [ ]:
# we load the rosetta data
rosetta_sim_df = pd.read_excel(r"C:\Users\jordant\PycharmProjects\LCMS_RiPP\rosetta_modeling\selected_plp_sequences.xlsx")
# we merge the rosetta_sim_df to the rosetta_comparison df using a merge on the sequence and target_sequence

agg_rosetta = rosetta_comparison.groupby("peptide_sequence_meta")["mod_ratio"].agg("mean")
merged_score_rosetta = pd.merge(agg_rosetta, rosetta_sim_df, left_on="peptide_sequence_meta", right_on="sequence", how="left")

import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
score_col_names = ["total_score", "energy_peptide", "fa_atr", "fa_dun", "fa_sol", "ligand_score_1X", "ligand_score_2X", "ligand_score_3X", "ligand_score_4X", "packstat", "sasa", "sc", "sc_int_area"]

results = []

# 2. Calculate Spearman correlation for each score
for feature in score_col_names:
    # Drop NaNs to ensure the correlation function doesn't crash
    valid_data = merged_score_rosetta[["mod_ratio", feature]].dropna()

    if len(valid_data) > 2:  # Need at least 3 points to correlate
        rho, p_val = stats.spearmanr(valid_data[feature], valid_data["mod_ratio"])
        results.append({
            'rosetta_metric': feature,
            'spearman_rho': rho,
            'p_value': p_val,
            'n_variants': len(valid_data)
        })

results_df = pd.DataFrame(results)
# 3. Correct for multiple hypothesis testing (Benjamini-Hochberg FDR)
results_df['fdr_adjusted_p'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

# Sort by the strongest absolute correlation
results_df['abs_rho'] = results_df['spearman_rho'].abs()
results_df = results_df.sort_values(by='abs_rho', ascending=False).drop(columns=['abs_rho'])

# View the top correlations
print(results_df.head(10))



In [ ]:
# we save the processed ratios df to a excel file for further analysis inside the summary_data_files folder
processed_ratios_df.to_excel("summary_data_files/96_well_plates_processed_ratios_df.xlsx", index=False)
# we save the point_mutant_comparison df to a excel file for further analysis inside the summary_data_files folder
point_mutant_comparison.to_excel("summary_data_files/96_well_plates_point_mutant_comparison.xlsx", index=False)
# we save the rosetta_comparison df to a excel file for further analysis inside the summary_data_files folder
rosetta_comparison.to_excel("summary_data_files/96_well_plates_rosetta_comparison.xlsx", index=False)

In [ ]:
point_mutant_agg = point_mutant_comparison.groupby(["maturase_name", "peptide_sequence_meta"])["mod_ratio"].agg("mean")
point_mutant_agg.to_excel("summary_data_files/96_well_plates_point_mutant_agg.xlsx")

In [ ]:
# we analyze some exceptions/weird cases by hand

with open(r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47\rosetta_top_47_precursor_5_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_5_PlpX-PlpY_TJ-C-599_24_TEV_master_pipeline_output.pkl", "rb") as fp:
    data_rosetta_peptide_05 = pickle.load(fp)


In [ ]:
data_rosetta_peptide_05["peak_object"][18].plot_peak()
data_rosetta_peptide_05["peak_object"][19].plot_peak()
data_rosetta_peptide_05["peak_object"][20].plot_peak()
data_rosetta_peptide_05["peak_object"][21].plot_peak()

In [4]:
# --- Re-review a single group in more detail ---
# Launches the exact same interactive review dashboard as running
# `python peak_reviewing_pipeline.py -i ... -r ...` from the terminal, except scoped to just one
# already-saved ReplicateGroupResults pickle instead of discovering every group from the input
# Excel. Saves back to the same file, like continuing a normal review session would.
#
# Runs in the background inside Jupyter (non-blocking) -- open the printed URL in your browser.
# Re-running this cell with the same port will fail if the previous server is still alive;
# restart the kernel or pass a different `port=` if you need to relaunch it.
from peak_reviewing_pipeline import review_single_group

#GROUP_PKL_PATH = r"D:\Spliceotide_Data_Analysis\output_reports_plates\point_mutants_precursor_15_HypX-HypY_TJ-C-689_24_TEV\point_mutants_precursor_15_HypX-HypY_TJ-C-689_24_TEV_manually_reviewed.pkl.gz"
#GROUP_PKL_PATH = r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_8_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_8_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz"
#GROUP_PKL_PATH = r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_10_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_10_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz"
#GROUP_PKL_PATH=r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_12_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_12_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz"
#GROUP_PKL_PATH=r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_23_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_23_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz"
#GROUP_PKL_PATH=r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_25_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_25_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz"
#GROUP_PKL_PATH=r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_28_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_28_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz"
#GROUP_PKL_PATH=r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_42_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_42_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz"
GROUP_PKL_PATH=r"D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_44_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_44_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz"



review_single_group(GROUP_PKL_PATH)

Found 1 group(s) ready for review.

Open the dashboard in your browser at: http://127.0.0.1:8051

Dash app running on http://127.0.0.1:8051/
[Loading] Group 1/1: rosetta_top_47_precursor_44_PlpX-PlpY_TJ-C-599_24_TEV
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_44_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_44_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz
Spectral data for 'plate_1_D8_rosetta_top_47' successfully loaded from cache.
[Loading] Group 1/1: rosetta_top_47_precursor_44_PlpX-PlpY_TJ-C-599_24_TEV
Successfully loaded ReplicateGroupResults from: D:\Spliceotide_Data_Analysis\output_reports_plates\rosetta_top_47_precursor_44_PlpX-PlpY_TJ-C-599_24_TEV\rosetta_top_47_precursor_44_PlpX-PlpY_TJ-C-599_24_TEV_manually_reviewed.pkl.gz
Spectral data for 'plate_1_D8_rosetta_top_47' successfully loaded from cache.
Spectral data for 'plate_1_H8_rosetta_top_47' successfully loaded from cache.Spectral dat